# Notebook 1: Data Exploration

**NYU Deep Learning Spring 2026 — Text-to-SVG Kaggle Competition**

### Purpose
Before training any model, we need to understand the data we are working with.
This notebook answers the following questions:

1. What do the prompts look like? How long are they?
2. What do the SVGs look like? How complex are they?
3. How long are the SVGs in characters — and what does that mean for token budget?
4. Which training examples should we keep vs. filter out?
5. Are there any invalid or malformed SVGs in the training set?

The answers to these questions will directly determine our `max_seq_length`,
our dataset size, and our training strategy in Notebook 2.

### Competition constraint reminders
- Output SVGs must be **≤ 8,000 characters**
- Canvas must be **256×256 pixels**
- Max **256 path elements**
- Only specific tags allowed (no scripts, no animation)
- Any invalid SVG scores **zero** — validity is a hard gate

## 0. Install dependencies

Run this cell once at the start of your Colab session.
`cairosvg` is the same SVG renderer the competition uses for scoring,
so we use it here to check renderability during EDA.

In [ ]:
# Install required packages.
# cairosvg: SVG renderer (same one used by the competition scorer)
# datasets: HuggingFace library for loading our uploaded train/test data
%pip install -q cairosvg datasets pandas matplotlib

## 1. Load the data

We uploaded `train.csv` and `test.csv` to a private HuggingFace dataset repo.
Loading from HuggingFace means we don't need to manually upload files to Colab
every session — we just authenticate once and stream the data.

**Setup required (one time only):**
1. Go to huggingface.co → Settings → Access Tokens → create a token with read access
2. In Colab: click the 🔑 key icon in the left sidebar → add a secret named `HF_TOKEN`
3. Your dataset repo path should be: `YOUR_HF_USERNAME/svg-competition-data`

In [ ]:
import os
from google.colab import userdata

# Load HuggingFace token from Colab secrets (never hardcode tokens in notebooks)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", "yes" if os.environ.get("HF_TOKEN") else "NO — check Colab secrets")

In [ ]:
import pandas as pd

# ── CHANGE THIS to your HuggingFace username ──────────────────────────────────
HF_DATASET_REPO = "YOUR_HF_USERNAME/svg-competition-data"
# ─────────────────────────────────────────────────────────────────────────────

# Load train and test CSVs directly from HuggingFace
# The data_files argument maps split names to filenames in the repo
from datasets import load_dataset

dataset = load_dataset(
    HF_DATASET_REPO,
    data_files={"train": "train.csv", "test": "test.csv"},
)

# Convert to pandas DataFrames for easier exploration
train_df = dataset["train"].to_pandas()
test_df  = dataset["test"].to_pandas()

print(f"Train rows : {len(train_df):,}")
print(f"Test rows  : {len(test_df):,}")
print(f"\nTrain columns: {list(train_df.columns)}")
print(f"Test columns : {list(test_df.columns)}")

## 2. Sanity check — look at a few rows

Always look at raw data before doing anything else.
This catches encoding issues, unexpected columns, or obvious anomalies early.

In [ ]:
# Display the first few rows of the training set
# We truncate the svg column so it doesn't flood the output
display_df = train_df.copy()
display_df["svg"] = display_df["svg"].str[:80] + "..."

display(display_df.head(5))

In [ ]:
# Print one full example so we can see the actual SVG structure
# We pick row index 0 — change this to browse different examples
EXAMPLE_IDX = 0

print("=" * 60)
print("PROMPT:")
print(train_df.loc[EXAMPLE_IDX, "prompt"])
print("\nSVG (first 500 chars):")
print(train_df.loc[EXAMPLE_IDX, "svg"][:500])
print("\nFull SVG length:", len(train_df.loc[EXAMPLE_IDX, "svg"]), "characters")

In [ ]:
# Render the example SVG inline so we can visually confirm it looks right
# This uses IPython's SVG display — same idea as the tutorial notebook
from IPython.display import SVG, display

display(SVG(data=train_df.loc[EXAMPLE_IDX, "svg"]))

## 3. Check for missing or malformed data

Before computing any statistics, we need to know if the dataset has
nulls or empty strings that would corrupt our analysis.

In [ ]:
# Count null values in each column
print("Null values in train:")
print(train_df.isnull().sum())

print("\nNull values in test:")
print(test_df.isnull().sum())

In [ ]:
# Check for empty strings (different from null)
empty_prompts = (train_df["prompt"].str.strip() == "").sum()
empty_svgs    = (train_df["svg"].str.strip() == "").sum()

print(f"Empty prompts : {empty_prompts}")
print(f"Empty SVGs    : {empty_svgs}")

In [ ]:
# Check that all SVGs actually start with '<svg'
# The competition requires valid XML with <svg as the root element
valid_start = train_df["svg"].str.lstrip().str.lower().str.startswith("<svg")
n_invalid_start = (~valid_start).sum()

print(f"SVGs that don't start with <svg: {n_invalid_start}")
if n_invalid_start > 0:
    print("\nExamples of non-<svg starts:")
    print(train_df.loc[~valid_start, "svg"].str[:100].head(3).values)

## 4. SVG length analysis

This is the most important section for training decisions.

**Why SVG length matters:**
- The model generates SVG as a sequence of text tokens
- `max_seq_length` in training caps how many tokens fit in one example
- SVG characters tokenize at roughly **3–4 characters per token** for SVG code
  (SVG uses many short numeric strings and repeated attribute names)
- So 8,000 characters ≈ 2,000–2,700 tokens
- The prompt adds ~50–150 tokens on top
- This means `max_seq_length=2048` covers most SVGs under ~6,000 chars

We'll plot the distribution and decide on a filter threshold.

In [ ]:
# Compute character length of every SVG in the training set
train_df["svg_len"] = train_df["svg"].str.len()

# Summary statistics
stats = train_df["svg_len"].describe(percentiles=[.25, .50, .75, .90, .95, .99])
print("SVG character length statistics:")
print(stats.round(0).astype(int))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left plot: full distribution
axes[0].hist(train_df["svg_len"], bins=100, color="steelblue", edgecolor="none")
axes[0].axvline(8000, color="red",    linestyle="--", label="Competition limit (8,000)")
axes[0].axvline(6000, color="orange", linestyle="--", label="Our filter threshold (6,000)")
axes[0].set_xlabel("SVG character length")
axes[0].set_ylabel("Number of examples")
axes[0].set_title("Full distribution of SVG lengths")
axes[0].legend()

# Right plot: zoomed in to 0–10,000 chars to see the bulk of the data
axes[1].hist(train_df[train_df["svg_len"] <= 10000]["svg_len"],
             bins=100, color="steelblue", edgecolor="none")
axes[1].axvline(8000, color="red",    linestyle="--", label="Competition limit (8,000)")
axes[1].axvline(6000, color="orange", linestyle="--", label="Our filter threshold (6,000)")
axes[1].set_xlabel("SVG character length")
axes[1].set_ylabel("Number of examples")
axes[1].set_title("Zoomed: lengths up to 10,000 chars")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# How many examples survive at different filter thresholds?
# This tells us how much training data we retain vs. discard.
thresholds = [3000, 4000, 5000, 6000, 7000, 8000]

print(f"{'Threshold':>12}  {'Rows kept':>10}  {'% of total':>10}")
print("-" * 38)
for t in thresholds:
    kept = (train_df["svg_len"] <= t).sum()
    pct  = 100 * kept / len(train_df)
    print(f"{t:>12,}  {kept:>10,}  {pct:>9.1f}%")

## 5. Estimate token lengths

Character count is a proxy, but what we actually care about is **token count**,
since that determines `max_seq_length`.

We load the actual tokenizer to measure this precisely on a sample.
We use a sample rather than the full 50k rows to save time.

In [ ]:
# Install transformers for tokenizer access
# We load just the tokenizer here — no GPU needed
%pip install -q transformers

In [ ]:
from transformers import AutoTokenizer

# We use Qwen2.5-1.5B — the model we plan to fine-tune.
# Loading only the tokenizer is fast and requires no GPU.
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"Tokenizer loaded: {MODEL_ID}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

In [ ]:
# This is the chat template we'll use during training.
# We define it here so token counting matches the actual training format.
#
# Format: system message + user prompt + assistant SVG response
# The model sees all three parts during training and learns to generate
# the assistant (SVG) portion given the system + user portions.

SYSTEM_PROMPT = (
    "You are an SVG code generator. "
    "Given a description, return only valid SVG code with a single root <svg> element. "
    "The canvas must be 256x256 pixels. Do not include any explanation."
)

def format_training_example(prompt: str, svg: str) -> str:
    """Format a (prompt, svg) pair into the chat template.
    
    We use the tokenizer's apply_chat_template method so the formatting
    exactly matches what the model expects. This is important — using
    a different format at training vs. inference time degrades quality.
    """
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": prompt},
        {"role": "assistant", "content": svg},
    ]
    # tokenize=False returns the formatted string without converting to token IDs
    return tokenizer.apply_chat_template(messages, tokenize=False)


# Test on one example to see what the model actually receives
example_text = format_training_example(
    prompt=train_df.loc[0, "prompt"],
    svg=train_df.loc[0, "svg"]
)
print(example_text[:600])
print("...")
print(f"\nTotal formatted length: {len(example_text)} characters")

In [ ]:
import numpy as np

# Sample 2,000 rows to estimate token length distribution.
# Tokenizing all 50k rows is slow on CPU — 2k gives a good estimate.
SAMPLE_N = 2000
sample_df = train_df.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)

token_lengths = []
for _, row in sample_df.iterrows():
    formatted = format_training_example(row["prompt"], row["svg"])
    # encode() returns a list of token IDs — its length is the token count
    n_tokens = len(tokenizer.encode(formatted))
    token_lengths.append(n_tokens)

token_lengths = np.array(token_lengths)

print("Token length statistics (prompt + SVG, formatted):")
for pct in [50, 75, 90, 95, 99, 100]:
    print(f"  p{pct:3d}: {np.percentile(token_lengths, pct):.0f} tokens")

In [ ]:
# Plot token length distribution with max_seq_length candidates marked
fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(token_lengths, bins=80, color="steelblue", edgecolor="none")
ax.axvline(2048, color="red",    linestyle="--", linewidth=1.5, label="2048 tokens")
ax.axvline(1536, color="orange", linestyle="--", linewidth=1.5, label="1536 tokens")
ax.axvline(1024, color="green",  linestyle="--", linewidth=1.5, label="1024 tokens")

ax.set_xlabel("Token count (prompt + SVG)")
ax.set_ylabel("Number of examples (sample)")
ax.set_title("Token length distribution — helps us pick max_seq_length")
ax.legend()
plt.tight_layout()
plt.show()

# Print what fraction of examples fit under each candidate max_seq_length
print("Fraction of sampled examples fitting within max_seq_length:")
for limit in [1024, 1536, 2048, 2560, 3072]:
    pct = 100 * (token_lengths <= limit).mean()
    print(f"  {limit:5d} tokens: {pct:.1f}% of examples fit")

## 6. Prompt analysis

Understanding the prompts tells us what kinds of descriptions the model
needs to handle. Are they short icon descriptions or longer scene descriptions?

In [ ]:
# Basic prompt length statistics
train_df["prompt_len"] = train_df["prompt"].str.len()
train_df["prompt_words"] = train_df["prompt"].str.split().str.len()

print("Prompt character length:")
print(train_df["prompt_len"].describe().round(1))
print("\nPrompt word count:")
print(train_df["prompt_words"].describe().round(1))

In [ ]:
# Print 10 random prompts to get a feel for the writing style and content
print("10 random training prompts:\n")
for i, prompt in enumerate(train_df["prompt"].sample(10, random_state=7).values):
    print(f"{i+1:2d}. {prompt}")

In [ ]:
# Check the test prompts too — are they similar in style to training?
# If they're very different, that's a signal we need to be careful about
# prompt formatting during inference.
print("10 random test prompts:\n")
for i, prompt in enumerate(test_df["prompt"].sample(10, random_state=7).values):
    print(f"{i+1:2d}. {prompt}")

## 7. SVG structure analysis

We want to understand what SVG elements are most common in the training data.
This helps us understand the complexity and style of SVGs we need to generate.

We also check path count — the competition enforces ≤ 256 paths.

In [ ]:
import xml.etree.ElementTree as ET
import re

def count_paths(svg_str: str) -> int:
    """Count the number of <path> elements in an SVG string.
    
    We use regex rather than XML parsing because some SVGs in the training
    set may be slightly malformed, and regex is more forgiving.
    """
    return len(re.findall(r"<path", svg_str, flags=re.IGNORECASE))


def count_element(svg_str: str, tag: str) -> int:
    """Count occurrences of a specific SVG tag."""
    return len(re.findall(f"<{tag}[\\s/>]", svg_str, flags=re.IGNORECASE))


# Compute path count on a sample (full dataset would take a while)
sample_for_structure = train_df.sample(n=2000, random_state=42)

path_counts = sample_for_structure["svg"].apply(count_paths)

print("Path element count per SVG:")
print(path_counts.describe().round(1))
print(f"\nExamples with > 256 paths: {(path_counts > 256).sum()}")

In [ ]:
# Count how often each SVG element type appears across the sample
# The allowed tags from the competition rules are our reference list
ALLOWED_TAGS = [
    "path", "rect", "circle", "ellipse", "line",
    "polyline", "polygon", "text", "g", "defs",
    "linearGradient", "radialGradient", "clipPath",
    "mask", "symbol", "use", "pattern", "marker", "filter"
]

tag_counts = {}
for tag in ALLOWED_TAGS:
    total = sample_for_structure["svg"].apply(
        lambda s: count_element(s, tag)
    ).sum()
    tag_counts[tag] = total

# Sort by frequency and display
tag_series = pd.Series(tag_counts).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
tag_series.plot(kind="bar", ax=ax, color="steelblue", edgecolor="none")
ax.set_xlabel("SVG element type")
ax.set_ylabel("Total occurrences in sample")
ax.set_title("Frequency of SVG element types in training data (2k sample)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Check for disallowed elements

The competition disallows: `script`, `animate`, `animateTransform`,
`animateMotion`, `set`, `foreignObject`, and any external references (href/src
pointing outside the SVG).

If the training data contains these, our model may learn to generate them
and score zero on those outputs. We should filter them out.

In [ ]:
DISALLOWED_TAGS = [
    "script",
    "animate",
    "animateTransform",
    "animateMotion",
    "foreignObject",
    "set",
]

print("Training examples containing disallowed elements:\n")
for tag in DISALLOWED_TAGS:
    has_tag = train_df["svg"].str.contains(f"<{tag}", case=False, regex=False)
    n = has_tag.sum()
    print(f"  <{tag}>: {n:,} examples ({100*n/len(train_df):.2f}%)")

In [ ]:
# Also check for external references — the competition disallows these
# External references look like href="http..." or src="http..."
has_external = train_df["svg"].str.contains(r'href=["\']http', case=False, regex=True)
n_external = has_external.sum()
print(f"Examples with external href references: {n_external:,}")

## 9. XML validity check

The competition requires SVGs to parse as valid XML.
We check what fraction of the training SVGs are valid XML right now.

Any training example that isn't valid XML is teaching the model to
generate invalid output — we should drop those.

In [ ]:
def is_valid_xml(svg_str: str) -> bool:
    """Return True if the string parses as valid XML.
    
    We use Python's built-in xml.etree.ElementTree which is strict —
    if it parses successfully, the SVG will pass the competition's
    XML validity check.
    """
    try:
        root = ET.fromstring(svg_str)
        # Also confirm the root tag is actually <svg>
        return root.tag.lower().endswith("svg")
    except ET.ParseError:
        return False


# Run on a sample — full 50k would be slow but you can remove the sample
# if you want exact numbers
validity_sample = train_df.sample(n=3000, random_state=42)
is_valid = validity_sample["svg"].apply(is_valid_xml)

n_valid   = is_valid.sum()
n_invalid = (~is_valid).sum()
print(f"Valid XML   : {n_valid:,} ({100*n_valid/len(validity_sample):.1f}%)")
print(f"Invalid XML : {n_invalid:,} ({100*n_invalid/len(validity_sample):.1f}%)")

In [ ]:
# If there are invalid ones, look at what's wrong with a few
invalid_examples = validity_sample[~is_valid]

if len(invalid_examples) > 0:
    print("Example invalid SVGs (first 200 chars each):\n")
    for i, (_, row) in enumerate(invalid_examples.head(3).iterrows()):
        print(f"Example {i+1}:")
        print(f"  Prompt: {row['prompt']}")
        print(f"  SVG start: {row['svg'][:200]}")
        print()

## 10. Renderability check

XML validity is necessary but not sufficient — a valid XML SVG can still
fail to render if it has missing required attributes or malformed path data.

We use `cairosvg` to actually render a sample of SVGs to PNG.
This is the same renderer the competition uses for scoring.

In [ ]:
import cairosvg
import io

def is_renderable(svg_str: str) -> bool:
    """Return True if cairosvg can render this SVG without error.
    
    We render to a bytes buffer (no file written to disk).
    If cairosvg raises any exception, the SVG fails the render gate
    and would score zero in the competition.
    """
    try:
        cairosvg.svg2png(
            bytestring=svg_str.encode("utf-8"),
            output_width=256,
            output_height=256
        )
        return True
    except Exception:
        return False


# Check a smaller sample — rendering is slower than XML parsing
render_sample = train_df.sample(n=500, random_state=42)
is_renderable_series = render_sample["svg"].apply(is_renderable)

n_render_ok   = is_renderable_series.sum()
n_render_fail = (~is_renderable_series).sum()
print(f"Renderable     : {n_render_ok:,} ({100*n_render_ok/len(render_sample):.1f}%)")
print(f"Render failures: {n_render_fail:,} ({100*n_render_fail/len(render_sample):.1f}%)")

## 11. Build the filtered training dataset

Based on the analysis above, we now apply all our filters in one place.
The goal is to keep examples that:

1. Have non-empty prompt and SVG
2. SVG starts with `<svg`
3. SVG is valid XML
4. SVG length ≤ our threshold (we use 6,000 chars as a conservative choice
   — update this based on what you saw in section 4)
5. Contains no disallowed elements

We skip the renderability check on the full dataset (too slow),
trusting that XML validity + disallowed-tag filtering is sufficient.

In [ ]:
# ── TUNE THIS based on the length distribution you saw above ─────────────────
# Lower threshold = faster training, more examples dropped
# Higher threshold = slower training, more examples kept
# Recommendation: choose the value where ~85-90% of examples fit
SVG_MAX_CHARS = 6000
# ─────────────────────────────────────────────────────────────────────────────

print(f"Starting with {len(train_df):,} training examples.")
print(f"Applying filters...\n")

filtered = train_df.copy()

# Filter 1: no nulls or empty strings
filtered = filtered[
    filtered["prompt"].notna() & (filtered["prompt"].str.strip() != "") &
    filtered["svg"].notna()    & (filtered["svg"].str.strip() != "")
]
print(f"After removing nulls/empty: {len(filtered):,}")

# Filter 2: SVG must start with <svg
filtered = filtered[
    filtered["svg"].str.lstrip().str.lower().str.startswith("<svg")
]
print(f"After requiring <svg start: {len(filtered):,}")

# Filter 3: SVG length under threshold
filtered = filtered[filtered["svg_len"] <= SVG_MAX_CHARS]
print(f"After length filter (≤{SVG_MAX_CHARS:,} chars): {len(filtered):,}")

# Filter 4: no disallowed elements
disallowed_pattern = "|".join([f"<{tag}" for tag in DISALLOWED_TAGS])
has_disallowed = filtered["svg"].str.contains(disallowed_pattern, case=False, regex=True)
filtered = filtered[~has_disallowed]
print(f"After removing disallowed elements: {len(filtered):,}")

# Filter 5: valid XML
# Note: this is slow on large datasets — we apply it last so the dataset
# is already small from the other filters
print("Checking XML validity (may take a minute)...")
xml_valid = filtered["svg"].apply(is_valid_xml)
filtered = filtered[xml_valid]
print(f"After XML validity check: {len(filtered):,}")

print(f"\nFinal filtered dataset: {len(filtered):,} examples")
print(f"Dropped: {len(train_df) - len(filtered):,} examples ({100*(len(train_df)-len(filtered))/len(train_df):.1f}%)")

## 12. Visual spot-check of filtered examples

Before saving, render a few examples from the filtered set to confirm
the remaining data looks visually reasonable.

In [ ]:
from IPython.display import HTML

# Display 6 random (prompt, SVG) pairs side by side as a visual sanity check
spot_check = filtered.sample(n=6, random_state=99).reset_index(drop=True)

html_parts = []
for _, row in spot_check.iterrows():
    html_parts.append(f"""
    <div style='display:inline-block; margin:10px; vertical-align:top; width:200px;'>
        <p style='font-size:11px; color:#555; margin:4px 0;'>{row['prompt'][:80]}</p>
        {row['svg']}
    </div>
    """)

display(HTML("".join(html_parts)))

## 13. Save the filtered dataset

We save the filtered training data back to HuggingFace as a new file
(`train_filtered.csv`) in the same dataset repo. Notebook 2 (training)
will load from this file rather than the raw `train.csv`.

We only keep the columns we need: `id`, `prompt`, `svg`.

In [ ]:
from huggingface_hub import HfApi
import io

# Keep only the columns needed for training
filtered_clean = filtered[["id", "prompt", "svg"]].reset_index(drop=True)

# Save locally first
LOCAL_FILTERED_PATH = "train_filtered.csv"
filtered_clean.to_csv(LOCAL_FILTERED_PATH, index=False)
print(f"Saved locally: {LOCAL_FILTERED_PATH} ({len(filtered_clean):,} rows)")

In [ ]:
# Upload the filtered file to HuggingFace
# This replaces any previous version of train_filtered.csv in the repo
api = HfApi(token=os.environ["HF_TOKEN"])

api.upload_file(
    path_or_fileobj=LOCAL_FILTERED_PATH,
    path_in_repo="train_filtered.csv",
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    commit_message=f"Add filtered training set ({len(filtered_clean):,} rows, max {SVG_MAX_CHARS} chars)"
)

print(f"Uploaded train_filtered.csv to {HF_DATASET_REPO}")

## 14. Summary and training decisions

Fill this in after running all cells above. This becomes your reference
for the hyperparameter choices in Notebook 2.

In [ ]:
# Auto-generate a summary of the decisions made in this notebook
p95_tokens = int(np.percentile(token_lengths, 95))
p99_tokens = int(np.percentile(token_lengths, 99))

# Round up p95 to nearest 256 (GPU memory is managed in chunks of 256 tokens)
recommended_seq_len = ((p95_tokens // 256) + 1) * 256

print("=" * 55)
print("TRAINING DECISIONS SUMMARY")
print("=" * 55)
print(f"  Total training examples (raw)    : {len(train_df):,}")
print(f"  Total training examples (filtered): {len(filtered_clean):,}")
print(f"  SVG max char filter applied       : {SVG_MAX_CHARS:,} chars")
print()
print(f"  Token length p95 (sample)         : {p95_tokens}")
print(f"  Token length p99 (sample)         : {p99_tokens}")
print(f"  Recommended max_seq_length        : {recommended_seq_len}")
print()
print(f"  Model to fine-tune  : Qwen/Qwen2.5-1.5B-Instruct")
print(f"  Training data file  : {HF_DATASET_REPO}/train_filtered.csv")
print("=" * 55)
print()
print("Copy max_seq_length into CONFIG in Notebook 2.")